# Week 4 — Baseline Model
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 4  

**Goal:** Train a baseline Linear Regression model, experiment with different training window sizes, and evaluate using R², MAPE, and MdAPE. Record results as "Linear Regression A" — the first experiment in an iterative modeling process.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error

print('Libraries loaded!')

Libraries loaded!


## 2. Load Cleaned Data
Loading the full cleaned dataset from Week 3, which includes all months (not just one fixed train/test split) so I can experiment with different training window sizes.

In [2]:
output_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\outputs'

df = pd.read_csv(output_folder + r'\cleaned_full.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])
df['YearMonth'] = df['CloseDate'].dt.to_period('M')

print(f'Full cleaned shape: {df.shape}')

Full cleaned shape: (340095, 13)


## 3. Define Features, Target, and Test Set
Per guidance, the test set must be the most recent full month (May 2026) so every intern is evaluated on the same market conditions.

In [3]:
feature_cols = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
    'LotSizeAcres', 'PropertyAge', 'GarageSpaces',
    'PoolPrivateYN', 'Latitude', 'Longitude'
]
target_col = 'ClosePrice'

test = df[df['YearMonth'] == '2026-05'].copy()

print(f'Test set: {len(test):,} rows (May 2026)')

Test set: 10,412 rows (May 2026)


## 4. Training Window Experiment
Since there's no universally optimal training window, I'm testing several window sizes (6, 12, 18, 24 months) immediately preceding the test month, and comparing performance. Features are normalized using StandardScaler, fit only on the training set to avoid leaking test-set information.

In [4]:
def evaluate_window(X_months):
    train_end = pd.Period('2026-04', freq='M')
    train_start = train_end - (X_months - 1)
    
    train = df[(df['YearMonth'] >= train_start) & (df['YearMonth'] <= train_end)].copy()
    
    X_train = train[feature_cols]
    y_train = train[target_col]
    X_test = test[feature_cols]
    y_test = test[target_col]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mdape = np.median(np.abs((y_test - y_pred) / y_test))
    
    return {'window_months': X_months, 'train_rows': len(train), 'r2': r2, 'mape': mape, 'mdape': mdape}

print('Function defined!')

Function defined!


## 5. Run the Window Size Comparison

In [5]:
window_sizes = [6, 12, 18, 24]
results = [evaluate_window(x) for x in window_sizes]

results_df = pd.DataFrame(results)
print(results_df)

   window_months  train_rows        r2      mape     mdape
0              6       52095  0.270303  0.566232  0.350275
1             12      113754  0.273287  0.552527  0.335536
2             18      165804  0.276159  0.530641  0.321462
3             24      228574  0.277575  0.518077  0.313050


## 6. Select Best Training Window

In [6]:
best = results_df.loc[results_df['r2'].idxmax()]

print(f"Best training window: {int(best['window_months'])} months")
print(f"R²:    {best['r2']:.4f}")
print(f"MAPE:  {best['mape']*100:.2f}%")
print(f"MdAPE: {best['mdape']*100:.2f}%")

Best training window: 24 months
R²:    0.2776
MAPE:  51.81%
MdAPE: 31.30%


## 7. Baseline Model Summary — Linear Regression A

**Decisions made:**
- Trained Linear Regression as the first baseline model, labeled "Linear Regression A"
- Features normalized with StandardScaler (fit on training data only, applied to test data) to prevent leakage
- Tested training windows of 6, 12, 18, and 24 months, all evaluated against the same fixed test month (May 2026)
- Selected [X]-month window as best based on highest R²

**Results:**
- Best training window: 24 months
- R²: 0.2776
- MAPE: 51.81%
- MdAPE: 31.30%

**Reasoning:**
The 24-month window performed best across all three metrics, though the difference 
between window sizes was modest (R² ranged from 0.270 to 0.278). Longer training 
windows likely helped by giving the model more examples to learn from, slightly 
outweighing the risk of including outdated market patterns.

The overall R² of 0.28 is low in absolute terms, which is expected for a Linear 
Regression baseline on this feature set. Linear Regression assumes linear relationships 
between features and price, but real estate pricing is driven by non-linear interactions 
(e.g., location effects, diminishing returns on square footage) that this model can't 
capture. This baseline establishes a performance floor — future iterations using 
tree-based models like XGBoost are expected to substantially improve R² and reduce 
MAPE/MdAPE by capturing these non-linear relationships.